# Experiment: Delta-Globin / HbA2 Induction Gate

Objective: decide whether `HBD` / delta-globin reactivation and HbA2 induction should become a Nakafa Lab therapy lane, assay benchmark, or rejected idea.

Success criteria:
- recover source-backed mouse, editing, molecule-screening, and human-observation evidence;
- separate HbA2 compensation biology from HbF reactivation biology;
- return a conservative decision without patient treatment advice.


In [ ]:
from __future__ import annotations

import json
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
PUBMED_DIR = ROOT / "data" / "literature" / "pubmed"
SELECTED_XML = PUBMED_DIR / "2026-04-27-delta-globin-hba2-selected-abstracts.xml"

SNAPSHOT_PATHS = {
    "delta_therapeutic": PUBMED_DIR
    / "2026-04-27-delta-globin-beta-thalassemia-therapeutic-search.json",
    "hba2_therapeutic_gap": PUBMED_DIR
    / "2026-04-27-hba2-beta-thalassemia-therapeutic-search.json",
    "hbd_promoter_strict_gap": PUBMED_DIR
    / "2026-04-27-hbd-promoter-delta-globin-gene-editing-search.json",
    "interferon_hba2": PUBMED_DIR
    / "2026-04-27-interferon-delta-globin-hba2-search.json",
    "drug_repurposing": PUBMED_DIR
    / "2026-04-27-delta-globin-drug-repurposing-search.json",
    "hbd_promoter_engineering": PUBMED_DIR
    / "2026-04-27-hbd-promoter-engineering-hba2-search.json",
    "hbd_cis_regulatory": PUBMED_DIR
    / "2026-04-27-hbd-cis-regulatory-elements-delta-globin-search.json",
    "hba2_induction_gap": PUBMED_DIR
    / "2026-04-27-hba2-induction-beta-hemoglobinopathies-search.json",
}

EVIDENCE_GROUPS = {
    "mouse_in_vivo_therapeutic": {"23872310"},
    "hbd_promoter_engineering": {"37265399", "40114594"},
    "pharmacologic_or_repurposing_signal": {"32528964", "40305292"},
    "human_high_hba2_observation": {"1995096", "1698102", "1380206"},
}

REQUIRED_READOUTS = [
    "HBD messenger RNA",
    "delta-globin protein",
    "HbA2 percentage or alpha2-delta2 assembly",
    "alpha/non-alpha globin balance",
    "HbF and F-cell percentage as separate comparators",
    "erythroid maturation",
    "viability, apoptosis, and mature red-cell hemolysis",
]

## Load Snapshot Rows

The zero-result snapshots are retained as query controls. They prevent later readers from mistaking one strict negative query for absence of the whole lane.


In [ ]:
def load_snapshot(path: Path) -> dict[str, Any]:
    """Read one compact PubMed JSON snapshot."""
    return json.loads(path.read_text())


def snapshot_summary(name: str, snapshot: dict[str, Any]) -> dict[str, object]:
    """Return a compact row for one PubMed snapshot."""
    results = snapshot.get("results")
    if not isinstance(results, list):
        results = []

    return {
        "lane": name,
        "query": snapshot.get("query"),
        "count": snapshot.get("count"),
        "top_pmids": [str(item.get("pmid")) for item in results[:3]],
    }


snapshots = {name: load_snapshot(path) for name, path in SNAPSHOT_PATHS.items()}
snapshot_rows = [
    snapshot_summary(name, snapshot) for name, snapshot in snapshots.items()
]
snapshot_rows

## Parse Selected Abstracts

The selected XML bundle covers mouse proof-of-concept, HBD promoter engineering, pharmacologic signals, and human high-HbA2 observations.


In [ ]:
def compact_text(element: ET.Element | None) -> str:
    """Collapse XML text content into one readable string."""
    if element is None:
        return ""

    return " ".join("".join(element.itertext()).split())


def parse_selected_articles(path: Path) -> dict[str, dict[str, str]]:
    """Parse selected PubMed XML into PMID-keyed metadata."""
    root = ET.parse(path).getroot()
    articles: dict[str, dict[str, str]] = {}

    for item in root.findall(".//PubmedArticle"):
        pmid = item.findtext(".//PMID") or ""
        if not pmid:
            continue

        abstract_parts = [
            compact_text(part) for part in item.findall(".//AbstractText")
        ]

        articles[pmid] = {
            "title": compact_text(item.find(".//ArticleTitle")),
            "journal": item.findtext(".//Journal/Title") or "",
            "year": item.findtext(".//JournalIssue/PubDate/Year") or "",
            "abstract_head": " ".join(abstract_parts)[:260],
        }

    return articles


selected_articles = parse_selected_articles(SELECTED_XML)
selected_article_count = len(selected_articles)
selected_article_count

## Evidence Gate

The lane is promoted only if it has in-vivo or engineered proof plus a realistic assay plan. Drug signals remain screening leads unless they are separated from cytotoxicity and marrow-suppression risk.


In [ ]:
def group_evidence_rows(
    groups: dict[str, set[str]],
    articles: dict[str, dict[str, str]],
) -> list[dict[str, object]]:
    """Summarize evidence groups covered by selected PMIDs."""
    rows: list[dict[str, object]] = []

    for group, pmids in groups.items():
        covered_pmids = sorted(pmid for pmid in pmids if pmid in articles)
        rows.append(
            {
                "group": group,
                "covered_pmids": covered_pmids,
                "covered_count": len(covered_pmids),
                "example_titles": [
                    articles[pmid]["title"] for pmid in covered_pmids[:2]
                ],
            }
        )

    return rows


def gate_decision(rows: list[dict[str, object]]) -> dict[str, object]:
    """Convert evidence coverage into a conservative research decision."""
    covered_groups = {
        str(row["group"])
        for row in rows
        if isinstance(row.get("covered_count"), int) and row["covered_count"] > 0
    }

    required_groups = {
        "mouse_in_vivo_therapeutic",
        "hbd_promoter_engineering",
        "pharmacologic_or_repurposing_signal",
    }
    has_required_support = required_groups.issubset(covered_groups)

    if not has_required_support:
        decision = "hold_for_more_source_retrieval"
    else:
        decision = "advanced_hba2_compensation_benchmark_not_first_panel_lead"

    return {
        "lane": "delta_globin_hba2_induction",
        "decision": decision,
        "covered_groups": sorted(covered_groups),
        "missing_required_groups": sorted(required_groups - covered_groups),
        "required_readouts": REQUIRED_READOUTS,
    }


evidence_rows = group_evidence_rows(EVIDENCE_GROUPS, selected_articles)
decision = gate_decision(evidence_rows)
{"evidence_rows": evidence_rows, "decision": decision}

## Interpretation

Delta-globin / HbA2 induction is not the same as HbF reactivation. It is a separate non-beta-globin compensation idea: increase delta-globin so alpha chains can assemble into HbA2.

The route is scientifically interesting, but not yet an affordable patient-facing lead. Current support is strongest for promoter engineering, mouse models, and screening hits. Drug hits such as palbociclib-class or pathway inhibitors must be treated as assay probes because marrow toxicity and broad pathway effects can erase any hemoglobin benefit.


In [ ]:
final_result = {
    "decision": decision["decision"],
    "why_it_matters": [
        "HbA2 is alpha2-delta2, so HBD activation could compensate for low beta-globin",
        "mouse and HBD-promoter engineering evidence support biological plausibility",
        "small-molecule screens show the pathway can be perturbed pharmacologically",
    ],
    "why_not_first_affordable_lead": [
        "clinical disease-modifying HbA2 induction is not established",
        "many hits are gene-editing, interferon-linked, or cytotoxicity-prone probes",
        "HbA2 must improve chain balance without hurting erythroid maturation",
    ],
    "next_repo_action": (
        "document as HbA2 compensation benchmark and assay expansion gate"
    ),
}
final_result

## Decision

Promote delta-globin / HbA2 induction to an advanced compensation benchmark and assay expansion gate.

Do not add it to the first affordable wet-lab quote panel as a treatment candidate. Use it as a future direction if a qualified lab can measure HBD, delta-globin protein, HbA2 assembly, globin-chain balance, maturation, and hemolysis together.
